# HW4 workflow

This notebook builds the feature panel, validates the model, and prepares test scores step by step. The daily backtest engine is imported from `hw4.py`.

In [16]:
from pathlib import Path
import importlib
import json
import sys

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV

# the directory
candidates = [Path.cwd(), Path.cwd() / 'Homeworks' / 'HW4' / 'Wen_Sun']
nb_dir = next((p.resolve() for p in candidates if (p / 'hw4.py').exists()), Path.cwd().resolve())
sys.path.insert(0, str(nb_dir))

import course_data as cd
import hw4
importlib.reload(cd)
importlib.reload(hw4)

In [2]:
# data
database = hw4.REPO_ROOT / 'Lectures' / 'modeling' / 'data' / 'equities_top2000.duckdb'
config = hw4.Config(database=database, output_dir=nb_dir / 'outputs')
print('Test period:', config.test_start, 'to', config.test_end)

Test period: 2016-04-01 to 2026-09-30


In [3]:
#tables: benchmark, universe (symbol,sector), 
# membership (universe_rank: size), fundamentals
benchmark, universe, membership, fundamentals = hw4.load_metadata(config.database)
calendar = pd.DatetimeIndex(benchmark.index)

display(benchmark.head(2))
display(universe.head(2))
display(membership.head(2))
display(fundamentals.head(2))

,adj_close
date,
2000-01-03,2002.11
2000-01-04,1925.41


,symbol,sector
0,A,Healthcare
1,AA,Basic Materials


,symbol,quarter_start,quarter_end,selection_date,universe_rank
0,SEEL,2000-01-01,2000-03-31,1999-12-31,1
1,MSFT,2000-01-01,2000-03-31,1999-12-31,2


,symbol,fiscal_date,period,filing_date,reported_currency,revenue,net_income,operating_income,total_assets,total_debt,cash_and_equivalents,free_cash_flow,operating_cash_flow,shares_outstanding
0,A,2000-01-31,quarter,2000-03-15,NaN,2.246000e+09,131000000.0,171000000.0,7.107000e+09,997000000.0,1.368000e+09,293000000.0,384000000.0,440000000.0
1,A,2000-04-30,quarter,2000-06-12,NaN,2.485000e+09,166000000.0,214000000.0,7.321000e+09,98000000.0,9.780000e+08,-169000000.0,-38000000.0,457000000.0


In [4]:
#last market session of every calendar month
signal_dates = hw4.month_end_dates(calendar, config.train_start, '2026-08-31')
print('Monthly signal dates:', len(signal_dates))
print('First/last:', signal_dates.min(), signal_dates.max())

#compute trailing price features
price_panel = hw4.load_price_features(config.database, signal_dates)
print('Price feature rows:', len(price_panel))
display(price_panel.head())

Monthly signal dates: 200
First/last: 2010-01-29 00:00:00 2026-08-31 00:00:00
Price feature rows: 549636


,symbol,date,adj_close,momentum_21,momentum_63,momentum_252_21,momentum_10,momentum_5_10_5,momentum_5_21_5,momentum_5_63_5,...,relative_volume_10,relative_volume_21,relative_volume_63,amihud_illiquidity_10,amihud_illiquidity_21,amihud_illiquidity_63,return_volume_corr_10,return_volume_corr_21,return_volume_corr_63,eligible
0,A,2010-01-29,17.730000,-0.076081,0.106742,0.633191,-0.100457,-0.060080,-0.045673,0.114195,...,0.326477,0.370291,0.526225,0.000180,0.000135,0.000164,-0.345624,0.003519,0.076436,True
1,AA,2010-01-29,25.100000,-0.205947,0.069450,0.958488,-0.194997,-0.194457,-0.179136,0.003786,...,-0.137760,-0.141690,0.201316,0.000034,0.000042,0.000053,-0.776325,-0.557156,-0.132762,True
2,AAIC,2010-01-29,3.640130,-0.021387,0.292808,3.286108,-0.066172,-0.068987,-0.045981,0.145786,...,-0.280462,-0.392525,-0.650482,0.258208,0.204872,0.203062,-0.224459,-0.151572,0.198172,True
3,AAL,2010-01-29,5.005957,0.094846,0.696485,-0.250387,-0.050090,-0.028485,0.001174,0.488075,...,0.403504,0.451738,0.762178,0.000468,0.000444,0.000869,0.128536,0.352551,0.318155,True
4,AAN,2010-01-29,27.860000,-0.017977,0.117978,NaN,-0.039310,-0.020415,-0.025457,0.069542,...,0.208599,0.193458,-0.145003,0.000722,0.000754,0.000890,-0.124064,0.258253,0.336494,True


In [5]:
#turn quarterly records into conservative availability-dated feature events
events = cd.financial_events(fundamentals)
panel = cd.join_financials(price_panel, events, universe)
panel = hw4.add_turnover_features(panel)
#eligible
panel = cd.apply_membership(panel, membership, calendar)
panel['eligible'] = panel['eligible'] & panel['is_universe_member']
panel = hw4.add_sector_neutral_factors(panel)

print('Panel rows:', len(panel))
print('Eligible signal rows:', int(panel.eligible.sum()))
display(panel.loc[panel.eligible, ['date', 'symbol', 'momentum_63', 'net_margin', 'sector', 'eligible']].head())

Panel rows: 549636
Eligible signal rows: 398144


,date,symbol,momentum_63,net_margin,sector,eligible
0,2010-01-29,A,0.106742,0.021422,Healthcare,True
2,2010-01-29,AAIC,0.292808,1.843004,Real Estate,True
3,2010-01-29,AAL,0.696485,-0.070021,Industrials,True
4,2010-01-29,AAN,0.117978,0.059373,Industrials,True
5,2010-01-29,AAON,0.136508,0.132343,Industrials,True


In [6]:
#a wide adjusted-close matrix indexed by the benchmark calendar
quotes = hw4.load_quote_matrix(config.database, calendar)
#first observed close on/after 90 calendar days
panel = hw4.make_labels_from_quotes(
    panel,
    quotes,
    calendar,
    holding_days=config.holding_days,
    max_exit_delay=config.max_exit_delay,
)

print('Quote matrix:', quotes.shape)
print('Labeled rows:', int(panel.target.notna().sum()))
display(panel[['date', 'symbol', 'entry_date', 'scheduled_exit', 'label_end', 'target']].head())

Quote matrix: (6726, 3849)
Labeled rows: 539779


,date,symbol,entry_date,scheduled_exit,label_end,target
0,2010-01-29,A,2010-02-01,2010-05-02,2010-05-03,0.262073
1,2010-01-29,AA,2010-02-01,2010-05-02,2010-05-03,-0.013662
2,2010-01-29,AAIC,2010-02-01,2010-05-02,2010-05-03,0.408145
3,2010-01-29,AAL,2010-02-01,2010-05-02,2010-05-03,0.319643
4,2010-01-29,AAN,2010-02-01,2010-05-02,2010-05-03,-0.187970


In [7]:
# availability checks: no future financials or future universe membership.
assert (panel.available_date.isna() | panel.available_date.le(panel.date)).all()
assert panel.loc[panel.eligible, 'universe_selection_date'].le(panel.loc[panel.eligible, 'date']).all()
print('Financial availability and universe timing checks passed.')

Financial availability and universe timing checks passed.


In [8]:
#generate validation splits with purging to avoid look-ahead bias.
development, folds = hw4.development_folds(panel, config)
print('Development rows:', len(development))
for year, (tr, va) in zip(config.validation_years, folds):
    print(f'{year}: train={len(tr)} validation={len(va)}')

Development rows: 142736
2013: train=66787 validation=23984
2014: train=90767 validation=23983


In [9]:
# Factor screening: average each factor's daily IC across training folds, then take its absolute value.
def factor_ic_summary(dev, folds, features):
    rows = []
    for feature in features:
        daily_ics = []
        for tr, va in folds:
            data = dev.iloc[tr]
            daily = data.groupby('date').apply(
                lambda x: x[feature].corr(x.model_target, method='spearman')
                if len(x) >= 5 and x[feature].nunique() > 1 and x.model_target.nunique() > 1
                else np.nan,
                include_groups=False,
            )
            daily_ics.append(daily)
        ic_series = pd.concat(daily_ics)
        rows.append({
            'feature': feature,
            'mean_ic': ic_series.mean(),
            'abs_mean_ic': abs(ic_series.mean()),
            'ic_dates': int(ic_series.notna().sum()),
        })
    return pd.DataFrame(rows).sort_values('abs_mean_ic', ascending=False).reset_index(drop=True)

factor_ic = factor_ic_summary(development, folds, hw4.NUMERIC_FEATURES)
display(factor_ic.round(4))

ic_threshold = 0.01
selected_numeric_features = factor_ic.loc[factor_ic.abs_mean_ic >= ic_threshold, 'feature'].tolist()
if not selected_numeric_features:
    selected_numeric_features = hw4.NUMERIC_FEATURES.copy()
selected_features = selected_numeric_features + ['sector']
print('Selected numeric features:', len(selected_numeric_features))
print(selected_numeric_features)

,feature,mean_ic,abs_mean_ic,ic_dates
0,operating_roa,0.0716,0.0716,80
1,sector_relative_operating_roa,0.0635,0.0635,80
2,sector_profitability_rank,0.0605,0.0605,80
3,sector_relative_fcf_to_assets,0.0439,0.0439,80
4,fcf_to_assets,0.0437,0.0437,80
...,...,...,...,...
58,sector_relative_return_volume_corr_21,-0.0067,0.0067,80
59,return_volume_corr_21,-0.0065,0.0065,80
60,momentum_21,-0.0064,0.0064,80
61,sector_relative_momentum_5_21_5,-0.0043,0.0043,80


Selected numeric features: 55
['operating_roa', 'sector_relative_operating_roa', 'sector_profitability_rank', 'sector_relative_fcf_to_assets', 'fcf_to_assets', 'volatility_63', 'sector_relative_volatility_63', 'sector_relative_net_margin', 'cash_to_assets', 'debt_to_assets', 'volatility_21', 'amihud_illiquidity_63', 'amihud_illiquidity_21', 'net_margin', 'sector_relative_cash_to_assets', 'sector_relative_volatility_21', 'amihud_illiquidity_10', 'sector_relative_amihud_illiquidity_63', 'volatility_10', 'sector_relative_amihud_illiquidity_21', 'drawdown_252', 'momentum_252_21', 'sector_relative_volatility_10', 'sector_relative_amihud_illiquidity_10', 'momentum_5_252_5', 'sector_relative_drawdown_252', 'sector_relative_momentum_252_21', 'sector_relative_debt_to_assets', 'sector_relative_momentum_5_252_5', 'momentum_10', 'turnover_21', 'turnover_63', 'momentum_5_10_5', 'sector_relative_momentum_10', 'turnover_10', 'sector_relative_momentum_5_10_5', 'relative_volume_63', 'sector_relative_re

In [10]:
# Tune each model family on the same purged folds, then compare their best scores.
tuned_models = {
    'OLS': (
        cd.make_pipeline(LinearRegression(), numeric_features=selected_numeric_features),
        {},
    ),
    'Ridge': (
        cd.make_pipeline(Ridge(), numeric_features=selected_numeric_features),
        {'model__alpha': list(config.alpha_grid)},
    ),
    'Lasso': (
        cd.make_pipeline(Lasso(max_iter=10000), numeric_features=selected_numeric_features),
        {'model__alpha': [0.0001, 0.001, 0.01]},
    ),
    'boosted trees': (
        cd.make_pipeline(
            HistGradientBoostingRegressor(early_stopping=False, random_state=42),
            numeric_features=selected_numeric_features,
        ),
        {
            'model__max_iter': [60],
            'model__max_leaf_nodes': [7, 15],
            'model__min_samples_leaf': [100],
            'model__l2_regularization': [1, 10],
        },
    ),
}

model_rows = []
best_estimators = {}
for name, (pipe, param_grid) in tuned_models.items():
    search = GridSearchCV(
        pipe,
        param_grid,
        cv=folds,
        scoring='neg_root_mean_squared_error',
        n_jobs=1,
        error_score='raise',
    )
    search.fit(development[selected_features], development.model_target)
    best_estimators[name] = search.best_estimator_
    model_rows.append({
        'model': name,
        'best_rmse': -search.best_score_,
        'best_params': str(search.best_params_),
    })

model_summary = pd.DataFrame(model_rows)

# evaluate each tuned best estimator on the same folds using ranking-oriented metrics.
for name, estimator in best_estimators.items():
    fold_metrics = []
    for fold_idx, (tr, va) in enumerate(folds, start=1):
        fitted = clone(estimator).fit(
            development.iloc[tr][selected_features],
            development.iloc[tr].model_target,
        )
        pred = fitted.predict(development.iloc[va][selected_features])
        eval_frame = development.iloc[va][['date', 'symbol']].copy()
        eval_frame['target'] = development.iloc[va].model_target
        fold_metrics.append(cd.prediction_metrics(eval_frame, pred).to_dict())
    fold_metrics = pd.DataFrame(fold_metrics)
    model_summary.loc[model_summary.model.eq(name), 'mean_rmse'] = fold_metrics.RMSE.mean()
    model_summary.loc[model_summary.model.eq(name), 'mean_mae'] = fold_metrics.MAE.mean()
    model_summary.loc[model_summary.model.eq(name), 'mean_r2'] = fold_metrics.R2.mean()
    model_summary.loc[model_summary.model.eq(name), 'mean_ic'] = fold_metrics.mean_daily_rank_IC.mean()

display(model_summary.round(4))

selectable = model_summary.loc[model_summary.mean_ic.notna()].copy()
# best_model_name = selectable.loc[selectable.mean_ic.idxmax(), 'model']
best_model_name = selectable.loc[
    selectable.mean_rmse.idxmin(), 'model'
]
chosen_model = best_estimators[best_model_name]
validation = model_summary
print('Selected model by mean daily rmse:', best_model_name)

,model,best_rmse,best_params,mean_rmse,mean_mae,mean_r2,mean_ic
0,OLS,0.2102,{},0.2102,0.1388,-0.2459,-0.0432
1,Ridge,0.1963,{'model__alpha': 5000.0},0.1963,0.1235,-0.0901,-0.0101
2,Lasso,0.1902,{'model__alpha': 0.01},0.1902,0.1186,-0.0294,-0.0225
3,boosted trees,0.1928,"{'model__l2_regularization': 10, 'model__max_i...",0.1928,0.1205,-0.0549,-0.0053


Selected model by mean daily rmse: Lasso


In [11]:
holdout_fit_date = calendar[calendar <= pd.Timestamp('2014-12-31')].max()
holdout_train = panel.loc[
    panel.eligible
    & panel.date.ge(config.train_start)
    & panel.date.le(holdout_fit_date)
    & panel.label_end.lt(holdout_fit_date)
    & panel.target.notna()
].copy()
holdout_score = panel.loc[
    panel.eligible & panel.date.between('2015-01-01', config.development_end)
].copy()
holdout_model = clone(chosen_model).fit(
    holdout_train[selected_features],
    hw4.clip_target(holdout_train.target, config.target_clip_lower, config.target_clip_upper),
)
holdout_pred = holdout_model.predict(holdout_score[selected_features])
holdout_eval = holdout_score.loc[holdout_score.target.notna()].copy()
holdout_eval['target'] = hw4.clip_target(
    holdout_eval.target, config.target_clip_lower, config.target_clip_upper
)
holdout_metrics = cd.prediction_metrics(
    holdout_eval,
    holdout_pred[holdout_score.target.notna().to_numpy()],
)
display(holdout_metrics)

RMSE                   0.198430
MAE                    0.142028
R2                    -0.156795
mean_daily_rank_IC    -0.096790
IC_dates              12.000000
dtype: float64

In [12]:
# Walk-forward test scoring using the selected features and model.
first_test_signal = calendar[calendar < pd.Timestamp(config.test_start)].max()
last_test_signal = hw4.month_end_dates(calendar, config.test_start, '2026-08-31').max()
fit_dates = hw4.quarter_end_dates(calendar, first_test_signal, last_test_signal)
if first_test_signal not in fit_dates:
    fit_dates = pd.DatetimeIndex([first_test_signal, *fit_dates[fit_dates > first_test_signal]])

score_parts = []
audit_parts = []
for i, fit_date in enumerate(fit_dates):
    next_fit = fit_dates[i + 1] if i + 1 < len(fit_dates) else pd.Timestamp('2026-09-01')
    predicted, audit = hw4.fit_score_quarter(
        panel, fit_date, next_fit, chosen_model, config, features=selected_features
    )
    score_parts.append(predicted)
    audit_parts.append(audit)

nonempty_score_parts = [part for part in score_parts if not part.empty]
test_scores = (
    pd.concat(nonempty_score_parts, ignore_index=True)
    if nonempty_score_parts
    else pd.DataFrame(columns=['date', 'symbol', 'score', 'fit_date'])
)
fit_audit = pd.DataFrame(audit_parts)
test_scores = test_scores.loc[test_scores.date.ge(first_test_signal)]
assert not test_scores.duplicated(['date', 'symbol']).any()
display(fit_audit)
display(test_scores.head())

,fit_date,rows,last_training_signal,last_label_end,first_scored_signal,last_scored_signal
0,2016-03-31,140739,2015-11-30,2016-02-29,2016-03-31,2016-05-31
1,2016-06-30,146724,2016-02-29,2016-05-31,2016-06-30,2016-08-31
2,2016-09-30,154693,2016-06-30,2016-09-29,2016-09-30,2016-11-30
3,2016-12-30,158685,2016-08-31,2016-12-05,2016-12-30,2017-02-28
4,2017-03-31,164684,2016-11-30,2017-03-01,2017-03-31,2017-05-31
5,2017-06-30,170668,2017-02-28,2017-06-02,2017-06-30,2017-08-31
6,2017-09-29,176648,2017-05-31,2017-08-30,2017-09-29,2017-11-30
7,2017-12-29,182632,2017-08-31,2017-11-30,2017-12-29,2018-02-28
8,2018-03-29,188612,2017-11-30,2018-03-01,2018-03-29,2018-05-31
9,2018-06-29,194607,2018-02-28,2018-05-31,2018-06-29,2018-08-31


,date,symbol,score,fit_date
0,2016-03-31,A,0.037075,2016-03-31
1,2016-03-31,AAIC,0.037292,2016-03-31
2,2016-03-31,AAL,0.037544,2016-03-31
3,2016-03-31,AAM,0.037734,2016-03-31
4,2016-03-31,AAN,0.036765,2016-03-31


In [13]:
sectors = pd.Series(
    panel.drop_duplicates('symbol').set_index('symbol').sector.fillna('Unknown')
)
strategy = hw4.MonthlySignalStrategy(test_scores, sectors, config)
result = hw4.SignalBacktester(quotes, benchmark.adj_close, strategy, config).run()
summary = hw4.performance_table(result, config)
display(summary)

,strategy,S&P 500 total return
final_value,2.293889e+06,4.402747e+06
total_return,1.293889e+00,3.402747e+00
CAGR,8.228427e-02,1.516195e-01
max_drawdown,-4.071055e-01,-3.379046e-01
annualized_volatility,3.962508e-01,1.781782e-01
Sharpe,3.420054e-01,7.725008e-01


In [14]:
display(result.history.tail())
display(result.fills.head())
display(result.open_lots.head())
display(result.diagnostics.event.value_counts())

,cash,holdings_value,nav,open_lots,fees,gross_traded,stale_marks,benchmark_nav
date,,,,,,,,
2026-09-24,98547.578739,2.192213e+06,2.290761e+06,94,0.000000,0.00000,44,4.432521e+06
2026-09-25,98547.578739,2.201754e+06,2.300301e+06,94,0.000000,0.00000,44,4.455183e+06
2026-09-28,98547.578739,2.181451e+06,2.279998e+06,94,0.000000,0.00000,44,4.420845e+06
2026-09-29,615202.355534,1.679862e+06,2.295064e+06,78,51.670645,516706.44744,44,4.413448e+06
2026-09-30,615202.355534,1.678687e+06,2.293889e+06,78,0.000000,0.00000,44,4.402747e+06


,date,signal_date,lot_id,symbol,side,price,units,notional,fee,cash_change,due_date,realized_pnl
0,2016-04-01,2016-03-31,0,AVTX,BUY,10512.000000,1.585331,16665.000167,1.6665,-16666.666667,2016-06-30,0.0
1,2016-04-01,2016-03-31,1,CONN,BUY,12.120000,1375.000014,16665.000167,1.6665,-16666.666667,2016-06-30,0.0
2,2016-04-01,2016-03-31,2,CRC,BUY,9.730000,1712.744108,16665.000167,1.6665,-16666.666667,2016-06-30,0.0
3,2016-04-01,2016-03-31,3,EGRX,BUY,40.669998,409.761519,16665.000167,1.6665,-16666.666667,2016-06-30,0.0
4,2016-04-01,2016-03-31,4,PBYI,BUY,29.190000,570.914703,16665.000167,1.6665,-16666.666667,2016-06-30,0.0


,lot_id,symbol,units,signal_date,entry_date,entry_price,entry_fee,due_date
0,1184,NEBCU,3429.211330,2021-06-30,2021-07-01,10.47,3.590384,2021-09-29
1,1220,PNNTG,1388.428559,2021-08-31,2021-09-01,25.29,3.511336,2021-11-30
2,1254,CFXA,132.348776,2021-10-29,2021-11-01,210.74,2.789118,2022-01-30
3,1271,CFXA,187.020313,2021-11-30,2021-12-01,181.81,3.400216,2022-03-01
4,1274,FMBIO,1239.145889,2021-11-30,2021-12-01,27.44,3.400216,2022-03-01


event
stale_mark                27010
stale_mark_over_limit     20747
rejected_missing_quote       31
Name: count, dtype: int64

In [ ]:
checks = hw4.validate_accounting(result, quotes, config)
print(json.dumps(checks, indent=2, default=str))

output_dir = config.output_dir
output_dir.mkdir(parents=True, exist_ok=True)
hw4.save_outputs(output_dir, result, test_scores, fit_audit, validation, config, checks)
hw4.plot_results(output_dir, result, config)
print('Saved outputs to:', output_dir)